# UdaPlay — Part 2: AI Research Agent

This notebook demonstrates the full **UdaPlay AI Research Agent** — a stateful agent that answers natural language questions about video games using:

- **RAG retrieval** — semantic search over the ChromaDB game knowledge base
- **Retrieval evaluation** — quality scoring to decide if RAG results are sufficient
- **Web search fallback** — Tavily API when the knowledge base has no answer
- **Tool use** — OpenAI function calling with `@tool` decorated Python functions
- **Session memory** — multi-turn conversation via `ShortTermMemory`
- **Evaluation** — `AgentEvaluator` for trajectory and response quality scoring

**Prerequisites:** Run Notebook 01 first to build the vector store. Set API keys in `.env`.

## 1. Setup

In [1]:
import sys
import os
import json
import glob

starter_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if starter_dir not in sys.path:
    sys.path.insert(0, starter_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(starter_dir, '..', '.env'))

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
CHROMA_OPENAI_API_KEY = os.getenv('CHROMA_OPENAI_API_KEY', OPENAI_API_KEY)
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

print('\u2713 Environment loaded')
print(f'  OPENAI_API_KEY:        {bool(OPENAI_API_KEY)}')
print(f'  CHROMA_OPENAI_API_KEY: {bool(CHROMA_OPENAI_API_KEY)}')
print(f'  TAVILY_API_KEY:        {bool(TAVILY_API_KEY)}')

✓ Environment loaded
  OPENAI_API_KEY:        True
  CHROMA_OPENAI_API_KEY: True
  TAVILY_API_KEY:        True


In [2]:
# Rebuild the vector store (re-index games)
from lib.documents import Document, Corpus
from lib.vector_db import VectorStoreManager
from lib.llm import LLM

games_dir = os.path.join(starter_dir, 'starter', 'games')
games = []
for filepath in sorted(glob.glob(os.path.join(games_dir, '*.json'))):
    with open(filepath, 'r', encoding='utf-8') as f:
        games.append(json.load(f))

def game_to_document(game: dict) -> Document:
    content = (
        f"{game['Name']} is a {game['Genre']} game published by {game['Publisher']} "
        f"on {game['Platform']} in {game['YearOfRelease']}. {game['Description']}"
    )
    return Document(content=content, metadata={
        'name': game['Name'],
        'platform': game['Platform'],
        'genre': game['Genre'],
        'publisher': game['Publisher'],
        'year': str(game['YearOfRelease']),
    })

corpus = Corpus([game_to_document(g) for g in games])
manager = VectorStoreManager(openai_api_key=CHROMA_OPENAI_API_KEY, chroma_path="./chroma_db")
store = manager.get_or_create_store('games')
store.add(corpus)

llm = LLM(model='gpt-4o-mini')

print(f'\u2713 Vector store ready \u2014 {len(games)} games indexed')
print(f'\u2713 ChromaDB persisting to {manager.chroma_path}')
print(f'\u2713 LLM ready: {llm.model}')

✓ Vector store ready — 15 games indexed
✓ ChromaDB persisting to ./chroma_db
✓ LLM ready: gpt-4o-mini


## 2. Define Agent Tools

We use the `@tool` decorator to wrap Python functions into OpenAI-compatible tool definitions. The agent can call these functions during its reasoning loop.

The agent follows a three-step workflow:
1. `retrieve_game` — search the internal knowledge base
2. `evaluate_retrieval` — **LLM-as-judge**: calls `gpt-4o-mini` to assess whether retrieved results are useful for answering the query, returning structured `useful` (bool) + `description`
3. `game_web_search` — fallback if the LLM judge returns `useful=False`

In [3]:
from lib.tooling import tool
from lib.llm import LLM as JudgeLLM
from lib.parsers import PydanticOutputParser
from pydantic import BaseModel as PydanticBaseModel, Field as PydanticField
from tavily import TavilyClient

tavily = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def retrieve_game(query: str) -> str:
    """Search the internal video game knowledge base for games matching a query.
    Returns a formatted string with the top matching game information."""
    results = store.query(query_texts=[query], n_results=3)
    docs = results.get('documents', [[]])[0]
    metadatas = results.get('metadatas', [[]])[0]
    distances = results.get('distances', [[]])[0]

    if not docs:
        return 'No games found in the knowledge base matching that query.'

    output = []
    for doc, meta, dist in zip(docs, metadatas, distances):
        similarity = 1 - dist
        output.append(
            f"Game: {meta.get('name')} | Platform: {meta.get('platform')} | "
            f"Genre: {meta.get('genre')} | Publisher: {meta.get('publisher')} | "
            f"Year: {meta.get('year')} | Similarity: {similarity:.3f}\n"
            f"Description: {doc[:300]}"
        )
    return '\n\n'.join(output)


class RetrievalJudgment(PydanticBaseModel):
    useful: bool = PydanticField(description="Whether the retrieved results are useful for answering the query")
    description: str = PydanticField(description="Brief explanation of the judgment")

_judge_llm = JudgeLLM(model='gpt-4o-mini')

@tool
def evaluate_retrieval(query: str, results: str) -> str:
    """Use an LLM judge to evaluate whether retrieved game results are useful for answering the query.
    Returns structured judgment with useful (boolean) and description (explanation)."""
    judge_prompt = f"""You are evaluating whether retrieved video game database results are useful for answering a user query.

User Query: {query}

Retrieved Results:
{results}

Evaluate whether these results contain enough relevant information to answer the query accurately.
Return useful=true if the results are helpful, useful=false if they are not relevant or insufficient."""

    response = _judge_llm.invoke(input=judge_prompt, response_format=RetrievalJudgment)
    parser = PydanticOutputParser(model_class=RetrievalJudgment)
    try:
        judgment = parser.parse(response)
    except Exception:
        judgment = RetrievalJudgment(
            useful=bool(results and len(results) > 100 and 'No games found' not in results),
            description="Fallback evaluation based on result content."
        )

    status = "relevant" if judgment.useful else "not_relevant"
    return f"{status}: useful={judgment.useful}. {judgment.description}"


@tool
def game_web_search(query: str) -> str:
    """Search the web for current video game information using Tavily.
    Use this when evaluate_retrieval returns useful=False."""
    results = tavily.search(f'video game: {query}', max_results=3)
    items = results.get('results', [])

    if not items:
        return 'No web results found.'

    output = []
    for item in items:
        output.append(
            f"Title: {item.get('title')}\n"
            f"URL: {item.get('url')}\n"
            f"Content: {item.get('content', '')[:400]}"
        )
    return '\n\n'.join(output)


print('\u2713 Tools defined:')
print(f'  - {retrieve_game}')
print(f'  - {evaluate_retrieval}')
print(f'  - {game_web_search}')

✓ Tools defined:
  - <Tool name=retrieve_game params=['query']>
  - <Tool name=evaluate_retrieval params=['query', 'results']>
  - <Tool name=game_web_search params=['query']>

## 3. Tool Demos (Direct Calls)

In [4]:
# Demo: retrieve_game
result = retrieve_game(query='racing game PlayStation')
print('retrieve_game("racing game PlayStation"):')
print(result)

retrieve_game("racing game PlayStation"):
Game: Gran Turismo | Platform: PlayStation 1 | Genre: Racing | Publisher: Sony Computer Entertainment | Year: 1997 | Similarity: 0.901
Description: Gran Turismo is a Racing game published by Sony Computer Entertainment on PlayStation 1 in 1997. A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre...

Game: Gran Turismo 5 | Platform: PlayStation 3 | Genre: Racing | Publisher: Sony Computer Entertainment | Year: 2010 | Similarity: 0.877
Description: Gran Turismo 5 is a Racing game published by Sony Computer Entertainment on PlayStation 3 in 2010. A comprehensive racing simulator featuring a vast selection of cars and tracks...

Game: Mario Kart 8 Deluxe | Platform: Nintendo Switch | Genre: Racing | Publisher: Nintendo | Year: 2017 | Similarity: 0.734
Description: Mario Kart 8 Deluxe is a Racing game published by Nintendo on Nintendo Switch in 2017. An enhanced version of Mario Kart 8 featuri

In [5]:
# Demo: evaluate_retrieval — relevant case
eval_result = evaluate_retrieval(query='racing game PlayStation', results=result)
print('evaluate_retrieval (relevant case):')
print(eval_result)

evaluate_retrieval (relevant case):
relevant: useful=True. The retrieved results contain detailed information about Gran Turismo racing games on PlayStation platforms, which is directly relevant to the query.

In [6]:
# Demo: evaluate_retrieval — not relevant case (query outside knowledge base)
low_sim_result = ("Game: Gran Turismo | Platform: PlayStation 1 | Genre: Racing | Year: 1997\n"
                  "Description: A realistic racing simulator...")
eval_result_low = evaluate_retrieval(query='upcoming games releasing in 2025', results=low_sim_result)
print('evaluate_retrieval (not relevant case):')
print(eval_result_low)

evaluate_retrieval (not relevant case):
not_relevant: useful=False. The retrieved results describe classic PlayStation racing games but contain no information about upcoming game releases in 2025. A web search would be more appropriate for this query.

In [7]:
# Demo: game_web_search (for content not in our knowledge base)
web_result = game_web_search(query='most anticipated games 2025')
print('game_web_search("most anticipated games 2025"):')
print(web_result[:800])

game_web_search("most anticipated games 2025"):
Title: The Most Anticipated Games of 2025
URL: https://www.ign.com/articles/most-anticipated-games-2025
Content: 2025 is shaping up to be a massive year for gaming with several high-profile releases on the horizon. Games like Grand Theft Auto VI, Monster Hunter Wilds, and Death Stranding 2 are among the most awaited titles. PlayStation fans are looking forward to Ghost of Yotei while Xbox is preparing...

Title: Best Upcoming Games 2025 | GameSpot
URL: https://www.gamespot.com/articles/best-upcoming-games-2025
Content: From action RPGs to open-world adventures, 2025 has something for every type of gamer. Our picks include Elden Ring's spiritual successor, a new FromSoftware IP, and several major sequels...


## 4. Build the Agent

In [8]:
from lib.agents import Agent

AGENT_INSTRUCTIONS = """You are UdaPlay, an expert video game research assistant.

You have access to three tools:
1. `retrieve_game` — searches an internal knowledge base of classic video games
2. `evaluate_retrieval` — uses an LLM judge to assess whether retrieved results are useful for answering the query
3. `game_web_search` — searches the web for current gaming news and information

Workflow:
1. Always call `retrieve_game` first to search the internal knowledge base.
2. Call `evaluate_retrieval` with the original query and the retrieve_game results.
3. If evaluate_retrieval returns useful=True, use those results to answer.
4. If evaluate_retrieval returns useful=False, call `game_web_search` as fallback.

Citation Requirements:
- When answering from the internal database, end with: Source: Internal Game Database
- When answering from web search, ALWAYS end your response with a numbered Sources section:
  Sources:
  1. [Article Title](https://url)
  2. [Another Source](https://url)
- Never provide a web-based answer without proper citations."""

agent = Agent(
    model_name='gpt-4o-mini',
    instructions=AGENT_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3,
)

print('\u2713 UdaPlay Agent initialized')
print(f'  Model: {agent.model_name}')
print(f'  Tools: {[t.name for t in agent.tools]}')
print(f'  Temperature: {agent.temperature}')
print(f'  Citations: required for web-based answers')

✓ UdaPlay Agent initialized
  Model: gpt-4o-mini
  Tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
  Temperature: 0.3
  Citations: required for web-based answers

## 5. Agent Runs — Example Queries

In [9]:
def run_agent(query: str, session_id: str = None, label: str = ''):
    print(f'{'='*60}')
    print(f'Query: "{query}"')
    if label:
        print(f'({label})')
    print('='*60)

    run = agent.invoke(query, session_id=session_id)
    final_state = run.get_final_state()
    messages = final_state.get('messages', [])

    answer = ''
    for msg in reversed(messages):
        if hasattr(msg, 'role') and msg.role == 'assistant' and msg.content:
            answer = msg.content
            break

    tools_used = []
    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tools_used.extend([tc.function.name for tc in msg.tool_calls])

    print(f'Tools used: {tools_used or ["none"]}')
    print(f'Total tokens: {final_state.get("total_tokens", 0)}')
    print(f'\nAnswer:\n{answer}')
    return run

In [10]:
# Query 1: Game in knowledge base — full retrieve→evaluate→answer flow
run1 = run_agent(
    'Who developed Gran Turismo and what platform is it on?',
    label='Expected: retrieve_game \u2192 evaluate_retrieval \u2192 answer'
)

Query: "Who developed Gran Turismo and what platform is it on?"
(Expected: retrieve_game → evaluate_retrieval → answer)
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Tools used: ['retrieve_game', 'evaluate_retrieval']
Total tokens: 521

Answer:
Gran Turismo was developed and published by **Sony Computer Entertainment**. It is available on the **PlayStation 1** and was released in 1997. It's a realistic racing simulator featuring a wide array of cars and tracks.


In [11]:
# Query 2: Another knowledge base query
run2 = run_agent(
    'What platform is Halo Infinite on and who published it?',
    label='Expected: retrieve_game \u2192 evaluate_retrieval \u2192 answer'
)

Query: "What platform is Halo Infinite on and who published it?"
(Expected: retrieve_game → evaluate_retrieval → answer)
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Tools used: ['retrieve_game', 'evaluate_retrieval']
Total tokens: 498

Answer:
Halo Infinite is available on the **Xbox Series X|S** and was published by **Xbox Game Studios**. It was released in 2021 and features Master Chief's return in a massive open-world environment on the ringworld Zeta Halo.


In [12]:
# Query 3: Beyond the dataset — web search fallback
run3 = run_agent(
    'What are the most anticipated video games coming out in 2025?',
    label='Expected: retrieve_game \u2192 evaluate_retrieval (not_relevant) \u2192 game_web_search \u2192 answer'
)

Query: "What are the most anticipated video games coming out in 2025?"
(Expected: retrieve_game → evaluate_retrieval (not_relevant) → game_web_search → answer)
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Tools used: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
Total tokens: 1043

Answer:
Based on current web search results, some of the most anticipated games of 2025 include:
- **Grand Theft Auto VI** — the long-awaited sequel from Rockstar Games
- **Monster Hunter Wilds** — Capcom’s next installment in the beloved series
- **Ghost of Yotei** — a PlayStation exclusive follow-up to Ghos

## 6. Multi-Turn Session Memory

In [13]:
SESSION_ID = 'udaplay-demo-session'

questions = [
    'What Pokemon games are in the knowledge base?',
    'Which of those is the newest?',
    'What platforms are Pokemon games on?',
]

session_runs = []
for i, q in enumerate(questions, 1):
    print(f'\n[Turn {i}] {q}')
    run = agent.invoke(q, session_id=SESSION_ID)
    final = run.get_final_state()
    messages = final.get('messages', [])
    answer = next((m.content for m in reversed(messages)
                   if hasattr(m, 'role') and m.role == 'assistant' and m.content), '')
    print(f'  \u2192 {answer[:200]}')
    session_runs.append(run)


[Turn 1] What Pokemon games are in the knowledge base?
  → There are two Pokémon games in the knowledge base: **Pokémon Gold and Silver** (Game Boy Color, 1999, Nintendo) and **Pokémon Ruby and Sapphire** (Game Boy Advance, 2002, Nintendo).

[Turn 2] Which of those is the newest?
  → The newest is **Pokémon Ruby and Sapphire**, released in 2002 for the Game Boy Advance — three years after Pokémon Gold and Silver (1999).

[Turn 3] What platforms are Pokemon games on?
  → The Pokémon games in the knowledge base span two Nintendo handheld platforms: **Game Boy Color** (Pokémon Gold and Silver, 1999) and **Game Boy Advance** (Pokémon Ruby and Sapphire, 2002).


In [14]:
# Show the accumulated session runs
all_runs = agent.get_session_runs(SESSION_ID)
print(f'Session "{SESSION_ID}" \u2014 {len(all_runs)} turns recorded')
print()
for i, run in enumerate(all_runs, 1):
    state = run.get_final_state()
    tokens = state.get('total_tokens', 0)
    duration = (run.end_timestamp - run.start_timestamp).total_seconds()
    print(f'Turn {i}: tokens={tokens}, duration={duration:.2f}s, run_id={run.run_id[:8]}...')

Session "udaplay-demo-session" — 3 turns recorded

Turn 1: tokens=612, duration=2.14s, run_id=c2a91f3b...
Turn 2: tokens=487, duration=1.41s, run_id=d7b84e2c...
Turn 3: tokens=531, duration=1.29s, run_id=a3c50f9d...


## 7. Agent Evaluation

In [15]:
from lib.evaluation import AgentEvaluator, TestCase

evaluator = AgentEvaluator()

# Define test cases
test_case_1 = TestCase(
    id='tc-001',
    description='Look up the publisher of a game in the knowledge base',
    user_query='Who developed Gran Turismo?',
    expected_tools=['retrieve_game', 'evaluate_retrieval'],
    reference_answer='Gran Turismo was developed and published by Sony Computer Entertainment.',
    max_steps=6
)

test_case_2 = TestCase(
    id='tc-002',
    description='Search for current gaming news not in the knowledge base',
    user_query='What are the most anticipated games of 2025?',
    expected_tools=['retrieve_game', 'evaluate_retrieval', 'game_web_search'],
    max_steps=10
)

print('\u2713 Test cases defined')
print(f'  TC-001: {test_case_1.description}')
print(f'  TC-002: {test_case_2.description}')

✓ Test cases defined
  TC-001: Look up the publisher of a game in the knowledge base
  TC-002: Search for current gaming news not in the knowledge base


In [16]:
import time

# Evaluate trajectory for run1 (Gran Turismo query)
traj_result = evaluator.evaluate_trajectory(test_case_1, run1)

print('=== Trajectory Evaluation: TC-001 ===')
print(f'Overall Score:         {traj_result.overall_score:.2f}')
print(f'Task Completed:        {traj_result.task_completion.task_completed}')
print(f'Steps Taken:           {traj_result.task_completion.steps_taken}')
print(f'Correct Tool Selected: {traj_result.tool_interaction.correct_tool_selected}')
print(f'Format Correct:        {traj_result.quality_control.format_correct}')
print(f'Total Tokens:          {traj_result.system_metrics.total_tokens}')
print(f'Execution Time:        {traj_result.system_metrics.execution_time:.2f}s')
print(f'Feedback:              {traj_result.feedback}')

=== Trajectory Evaluation: TC-001 ===
Overall Score:         1.00
Task Completed:        True
Steps Taken:           5
Correct Tool Selected: True
Format Correct:        True
Total Tokens:          521
Execution Time:        2.31s
Feedback:              Trajectory: 5 steps, Tools used: ['retrieve_game', 'evaluate_retrieval'], Expected: ['retrieve_game', 'evaluate_retrieval']


In [17]:
# Evaluate final response quality (LLM-as-judge)
final_state_1 = run1.get_final_state()
messages_1 = final_state_1.get('messages', [])
answer_1 = next((m.content for m in reversed(messages_1)
                 if hasattr(m, 'role') and m.role == 'assistant' and m.content), '')

response_result = evaluator.evaluate_final_response(
    test_case=test_case_1,
    agent_response=answer_1,
    execution_time=2.31,
    total_tokens=final_state_1.get('total_tokens', 0)
)

print('=== Final Response Evaluation: TC-001 ===')
print(f'Overall Score:         {response_result.overall_score:.2f}')
print(f'Task Completed:        {response_result.task_completion.task_completed}')
print(f'Format Correct:        {response_result.quality_control.format_correct}')
print(f'Instructions Followed: {response_result.quality_control.instructions_followed}')
print(f'Cost Estimate:         ${response_result.system_metrics.cost_estimate:.6f}')
print(f'\nFeedback: {response_result.feedback}')

=== Final Response Evaluation: TC-001 ===
Overall Score:         1.00
Task Completed:        True
Format Correct:        True
Instructions Followed: True
Cost Estimate:         $0.000196

Feedback: The agent correctly identified Sony Computer Entertainment as the developer of Gran Turismo and accurately stated the platform (PlayStation 1, 1997). The retrieve→evaluate workflow was followed correctly.


## 8. Performance Summary

In [18]:
all_query_runs = [run1, run2, run3]
queries = [
    'Who developed Gran Turismo and what platform is it on?',
    'What platform is Halo Infinite on and who published it?',
    'What are the most anticipated video games coming out in 2025?',
]

print('=== UdaPlay Agent Performance Summary ===')
print(f'{'Query':<55} {'Tokens':>7} {'Time':>6} {'Tools Used'}')
print('-' * 100)

total_tokens = 0
for q, run in zip(queries, all_query_runs):
    state = run.get_final_state()
    tokens = state.get('total_tokens', 0)
    total_tokens += tokens
    duration = (run.end_timestamp - run.start_timestamp).total_seconds()
    tools = []
    for msg in state.get('messages', []):
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tools.extend([tc.function.name for tc in msg.tool_calls])
    print(f'{q[:54]:<55} {tokens:>7} {duration:>5.2f}s  {tools}')

print('-' * 100)
print(f'Total tokens across all queries: {total_tokens}')
print(f'Estimated cost: ${total_tokens * 0.000000375:.6f}')

=== UdaPlay Agent Performance Summary ===
Query                                                    Tokens   Time Tools Used
----------------------------------------------------------------------------------------------------
Who developed Gran Turismo and what platform is it on?      521   2.31s  ['retrieve_game', 'evaluate_retrieval']
What platform is Halo Infinite on and who published it?     498   1.87s  ['retrieve_game', 'evaluate_retrieval']
What are the most anticipated video games coming out in     1043   3.21s  ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
----------------------------------------------------------------------------------------------------
Total tokens across all queries: 2062
Estimated cost: $0.000773


## Summary

| Feature | Implementation |
|---|---|
| **Tool definition** | `@tool` decorator → OpenAI function schema |
| **RAG retrieval** | `retrieve_game()` → `VectorStore.query()` |
| **Retrieval evaluation** | `evaluate_retrieval()` → LLM-as-judge (`gpt-4o-mini`) returns `useful` bool + `description` |
| **Web fallback** | `game_web_search()` → Tavily API (when judge returns `useful=False`) |
| **Citations** | Web answers include numbered `Sources:` section with title + URL |
| **Agent loop** | `Agent` class → state machine (message_prep → llm → tool_executor) |
| **Session memory** | `ShortTermMemory` stores `Run` objects per session |
| **Evaluation** | `AgentEvaluator` — trajectory, single-step, and LLM-as-judge |
| **Multi-turn** | Same `session_id` accumulates conversation across turns |